# release16_bull_reentry_participation_objective_replay_v1

## Governance

- Baselines: canonical `soxlbacktest_policy_reimplementation_v1`; direct comparison `release16_release_gate_bc_official_replay_v1`; handoff source `release16_official_replay_candidate_handoff_v1`.
- Role: `diagnose` training-objective repair before any retained actor.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Accepted trade-off: improve underparticipation/release recovery only if all official walk-forward folds keep `relative_return >= 30pp` and `MaxDD <= 20%`; strict `<=15%` is retained as risk context.
- Changed layer: `target`, `model_selection`, `official_replay`.
- External reference: converted SOXLBacktest qlib buy/sell baseline and release16 official replay candidate handoff.
- Hypothesis: adding a fold-local bull-reentry participation floor to the current release16 distillation contract can improve `wf_2020` and `wf_2023_2024` return gaps without breaking `wf_2022` drawdown.
- Acceptance criteria: target-scope audit passes; target lift rows are train-local; each candidate is replayed on all three folds; full gate means all folds pass `+30pp / <=20% MaxDD`; no actor is promoted or retained.
- Expected failure mode: bull-reentry rows may increase trend participation but either leave `wf_2020` below the return floor or raise MaxDD, so the next step would move to representation/checkpoint objective rather than more target lifting.
- Comparison-table row key: `release16_bull_reentry_participation_objective_replay_v1`.
- Closest historical success case: `release16_release_gate_bc_weight_screen_v1` fixed `wf_2023_2024` release-gate BC slice accuracy to 0.818.
- Closest historical failure case: `bull_reentry_three_fold_checkpoint_metric_replay_v1` passed some folds but failed `wf_2020` return, while `release16_release_gate_bc_official_replay_v1` improved release fit but failed official replay gates.
- Why this is not a duplicate: this notebook applies bull-reentry target variants to the current release16 distillation contract, uses the release64 selected weighting, and scores official warm-start daily path metrics in the latest handoff context.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.checkpoint_daily_export import build_checkpoint_candidate_daily_export, metrics_from_official_backtest
from src.soxl_quant.app.state_objective_bc import StateObjectiveBCSampleWeightConfig, StateObjectiveBCTrainingConfig, fit_state_objective_bc_actor_from_frame
from src.soxl_quant.app.state_objective_training_targets import (
    BullReentryParticipationTargetConfig,
    apply_bull_reentry_participation_target_gate,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "release16_bull_reentry_participation_objective_replay_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_TARGET_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
SOURCE_REPLAY_CARD = ROOT / "runtime/model_selection/card/release16_release_gate_bc_official_replay_v1"
HANDOFF_CARD = ROOT / "runtime/model_selection/card/release16_official_replay_candidate_handoff_v1"
TARGET_CONTRACT = SOURCE_TARGET_CARD / "distillation_target_contract.csv"
REFERENCE_METRICS = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_metrics.csv"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
MAX_DD_PCT = 20.0
STRICT_MAX_DD_PCT = 15.0
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "official_daily_paths"
DAILY_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED)
SELECTED_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=6.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=64.0,
    objective_weight_power=0.0,
    class_balanced_beta=0.99,
    weight_cap=80.0,
)
TARGET_CONFIGS: dict[str, BullReentryParticipationTargetConfig | None] = {
    "release64_current_target": None,
    "bull55_hold3": BullReentryParticipationTargetConfig(
        enabled=True,
        min_target_position=0.55,
        max_target_position=0.70,
        objective_weight_bonus=4.0,
        max_objective_weight=30.0,
        include_release_gate=True,
        forward_hold_days=3,
        exclude_downtrend=True,
        exclude_drawdown_control=True,
    ),
    "bull65_hold3": BullReentryParticipationTargetConfig(
        enabled=True,
        min_target_position=0.65,
        max_target_position=0.80,
        objective_weight_bonus=5.0,
        max_objective_weight=30.0,
        include_release_gate=True,
        forward_hold_days=3,
        exclude_downtrend=True,
        exclude_drawdown_control=True,
    ),
    "bull70_hold5": BullReentryParticipationTargetConfig(
        enabled=True,
        min_target_position=0.70,
        max_target_position=0.85,
        objective_weight_bonus=5.0,
        max_objective_weight=30.0,
        include_release_gate=True,
        forward_hold_days=5,
        exclude_downtrend=True,
        exclude_drawdown_control=True,
    ),
}
WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}
FOLDS = [
    {
        "name": spec["target_fold"],
        "train_start": spec["train"][0],
        "train_end": spec["train"][1],
        "eval_start": spec["test"][0],
        "eval_end": spec["test"][1],
    }
    for spec in WALK_FORWARD_WINDOWS.values()
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str], *, fold: str, candidate: str) -> dict[str, Any]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = metrics_from_official_backtest(split_metrics=dict(split_metrics), backtest_result=bt)
    export = build_checkpoint_candidate_daily_export(
        backtest_result=bt,
        price_series=price_series,
        output_dir=DAILY_DIR,
        fold=fold,
        candidate=candidate,
        candidate_kind="bc_initialization",
        split_metrics=dict(split_metrics),
        root=ROOT,
    )
    for source_key, target_key in {
        "test_return": "return",
        "test_max_dd": "max_dd",
        "test_avg_pos": "avg_pos",
        "test_buy_and_hold_return": "buy_and_hold_return",
        "test_relative_return": "relative_return",
    }.items():
        if source_key in export.row:
            out[target_key] = float(export.row[source_key])
    out.update({k: v for k, v in export.row.items() if k not in out})
    return out

print("root", ROOT)
print("card", CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_bull_reentry_participation_objective_replay_v1


In [2]:

profile = load_profile(PROFILE_NAME)
source_targets = pd.read_csv(TARGET_CONTRACT)
source_targets["datetime"] = pd.to_datetime(source_targets["datetime"]).dt.normalize()
for col in (
    "training_label_allowed",
    "is_rebound_cluster",
    "is_downtrend_state",
    "release_gate_state",
    "drawdown_control_state",
):
    if col in source_targets.columns:
        source_targets[col] = normalize_bool_series(source_targets[col])

handoff = load_json(HANDOFF_CARD / "training_objective_handoff.json")
source_replay_summary = load_json(SOURCE_REPLAY_CARD / "summary.json")
retention_summary = load_json(RETENTION_GATE).get("summary", {})

reference_metrics = pd.read_csv(REFERENCE_METRICS)
reference_rows = reference_metrics[
    reference_metrics["candidate"].isin(["legacy_default", "wf_pass_ytd18_15_15_02"])
    & reference_metrics["window"].isin(WALK_FORWARD_WINDOWS)
    & reference_metrics["split_type"].eq("walk_forward_test")
].copy()
reference_rows["relative_return_pct"] = reference_rows["rulebase_return_pct"] - reference_rows["soxl_buyhold_return_pct"]
reference_rows.to_csv(CARD / "soxlbacktest_reference_metrics.csv", index=False)

config_rows = []
for candidate, cfg in TARGET_CONFIGS.items():
    if cfg is None:
        config_rows.append({"candidate": candidate, "uses_bull_reentry": False})
    else:
        config_rows.append({"candidate": candidate, "uses_bull_reentry": True, **asdict(cfg)})
config_grid = pd.DataFrame(config_rows)
config_grid.to_csv(CARD / "target_config_grid.csv", index=False)
pd.DataFrame([{"config": "release64_no_objective", **asdict(SELECTED_WEIGHT_CONFIG)}]).to_csv(CARD / "bc_weight_config.csv", index=False)
print("profile", profile.name)
print("handoff", handoff)
display(config_grid)
display(reference_rows[["candidate", "window", "rulebase_return_pct", "rulebase_max_drawdown_pct", "relative_return_pct", "average_position_pct"]])


profile sparse_release_bc_ppo_screen_v1
handoff {'decision': 'train_rebound_participation_objective', 'input_counts': {'feature_groups': 9, 'poor_window_clusters': 8, 'walkforward_candidates': 11}, 'objective_terms': ['walkforward_relative_return_gap_penalty', 'walkforward_drawdown_gap_penalty', 'high_vol_rebound_participation_floor'], 'primary_underparticipation_cluster': {'candidate_id': 'release16_drawdown_upside_checkpoint_replay_v1::bc_initialization', 'cluster_id': 'wf_2020', 'dominant_failure_type': 'underparticipation', 'max_dd_gap': 0.0, 'mean_excess_return': -0.4270525146105939, 'mean_relative_gap': 30.427052514610594}, 'promotion': 'blocked', 'risk_gap': {'max_dd_gap': 4.33, 'observed_max_dd': 19.33, 'observed_min_relative_return': -0.43, 'relative_gap': 30.43, 'target_max_dd': 15.0, 'target_relative_return': 30.0, 'worst_window': 'wf_2020'}, 'router_cluster_delta_max': 0.0, 'selected_diagnostic_candidate': 'release16_drawdown_upside_checkpoint_replay_v1::bc_initialization',

,candidate,uses_bull_reentry,enabled,min_target_position,max_target_position,min_lift,objective_weight_bonus,max_objective_weight,include_release_gate,forward_hold_days,action_values,forward_action_values,stop_action_values,group_cols,target_scope,require_train_window,exclude_downtrend,exclude_drawdown_control
0,release64_current_target,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bull55_hold3,True,True,0.55,0.70,0.01,4.0,30.0,True,3.0,"(BUY,)","(HOLD,)","(SELL, TAKE_PROFIT)","(fold, window)",fold_local,True,True,True
2,bull65_hold3,True,True,0.65,0.80,0.01,5.0,30.0,True,3.0,"(BUY,)","(HOLD,)","(SELL, TAKE_PROFIT)","(fold, window)",fold_local,True,True,True
3,bull70_hold5,True,True,0.70,0.85,0.01,5.0,30.0,True,5.0,"(BUY,)","(HOLD,)","(SELL, TAKE_PROFIT)","(fold, window)",fold_local,True,True,True


,candidate,window,rulebase_return_pct,rulebase_max_drawdown_pct,relative_return_pct,average_position_pct
0,legacy_default,wf_2020,220.777205,54.275144,161.001897,79.102332
1,legacy_default,wf_2022,-79.396371,84.918053,7.191701,86.692988
2,legacy_default,wf_2023_2024,2.935118,32.461810,5.538546,31.043492
12,wf_pass_ytd18_15_15_02,wf_2020,29.156603,12.785207,-30.618705,17.101107
13,wf_pass_ytd18_15_15_02,wf_2022,-16.044551,18.514585,70.543521,10.231793
14,wf_pass_ytd18_15_15_02,wf_2023_2024,4.505779,1.061804,7.109208,1.322063


In [3]:

target_variants: dict[str, pd.DataFrame] = {}
target_summary_rows: list[dict[str, Any]] = []
scope_frames: list[pd.DataFrame] = []
lift_frames: list[pd.DataFrame] = []

for candidate, cfg in TARGET_CONFIGS.items():
    revised = source_targets.copy() if cfg is None else apply_bull_reentry_participation_target_gate(source_targets, cfg)
    revised["candidate"] = candidate
    target_variants[candidate] = revised
    variant_path = CARD / f"{candidate}_target_contract.csv"
    revised.to_csv(variant_path, index=False)

    bull_state = revised.get("bull_reentry_target_state", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    bull_forward = revised.get("bull_reentry_forward_ready", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    release_state = revised.get("release_gate_state", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    rebound_state = revised.get("is_rebound_cluster", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    downtrend_state = revised.get("is_downtrend_state", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    drawdown_state = revised.get("drawdown_control_state", pd.Series(False, index=revised.index)).fillna(False).astype(bool)
    allowed = revised.get("training_label_allowed", pd.Series(True, index=revised.index)).fillna(False).astype(bool)

    scope = audit_state_objective_target_scope(revised[allowed].copy(), folds=FOLDS)
    scope.insert(0, "candidate", candidate)
    scope_frames.append(scope)
    lift = audit_state_objective_target_lift(
        revised[allowed].copy(),
        group_cols=("fold", "window"),
        baseline_risk_position_col="risk_target_position_original" if "risk_target_position_original" in revised.columns else "risk_target_position",
    )
    lift.insert(0, "candidate", candidate)
    lift_frames.append(lift)
    target_summary_rows.append(
        {
            "candidate": candidate,
            "target_contract": str(variant_path.relative_to(ROOT)),
            "rows": int(len(revised)),
            "allowed_rows": int(allowed.sum()),
            "bull_reentry_target_rows": int(bull_state.sum()),
            "bull_reentry_forward_rows": int((bull_state & bull_forward).sum()),
            "release_gate_rows": int(release_state.sum()),
            "rebound_rows": int(rebound_state.sum()),
            "downtrend_rows": int(downtrend_state.sum()),
            "drawdown_control_rows": int(drawdown_state.sum()),
            "mean_risk_target_position": float(pd.to_numeric(revised["risk_target_position"], errors="coerce").mean()),
            "max_risk_target_position": float(pd.to_numeric(revised["risk_target_position"], errors="coerce").max()),
            "scope_decision": summarize_state_objective_target_scope_audit(scope)["decision"],
            "scope_training_allowed": bool(summarize_state_objective_target_scope_audit(scope)["training_allowed"]),
            "lift_decision": summarize_state_objective_target_lift_audit(lift)["decision"],
            "lift_training_allowed": bool(summarize_state_objective_target_lift_audit(lift)["training_allowed"]),
        }
    )

target_summary = pd.DataFrame(target_summary_rows)
scope_audit = pd.concat(scope_frames, ignore_index=True) if scope_frames else pd.DataFrame()
lift_audit = pd.concat(lift_frames, ignore_index=True) if lift_frames else pd.DataFrame()
target_summary.to_csv(CARD / "target_variant_summary.csv", index=False)
scope_audit.to_csv(CARD / "target_scope_audit.csv", index=False)
lift_audit.to_csv(CARD / "target_lift_audit.csv", index=False)

display(target_summary)
display(scope_audit)
display(lift_audit)


,candidate,target_contract,rows,allowed_rows,bull_reentry_target_rows,bull_reentry_forward_rows,release_gate_rows,rebound_rows,downtrend_rows,drawdown_control_rows,mean_risk_target_position,max_risk_target_position,scope_decision,scope_training_allowed,lift_decision,lift_training_allowed
0,release64_current_target,runtime/model_selection/card/release16_bull_re...,3731,3731,0,0,62,220,1585,873,0.309136,1.0,state_objective_targets_training_ready,True,state_objective_target_lift_training_ready,True
1,bull55_hold3,runtime/model_selection/card/release16_bull_re...,3731,3731,21,20,62,240,1585,873,0.311006,1.0,state_objective_targets_training_ready,True,state_objective_target_lift_training_ready,True
2,bull65_hold3,runtime/model_selection/card/release16_bull_re...,3731,3731,21,20,62,240,1585,873,0.311569,1.0,state_objective_targets_training_ready,True,state_objective_target_lift_training_ready,True
3,bull70_hold5,runtime/model_selection/card/release16_bull_re...,3731,3731,68,54,62,274,1585,873,0.314181,1.0,state_objective_targets_training_ready,True,state_objective_target_lift_training_ready,True


,candidate,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue,train_start,train_end,eval_start,eval_end
0,release64_current_target,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2218,0,252,True,,2010-03-11,2018-12-31,2020-01-01,2020-12-31
1,release64_current_target,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,253,0,252,True,,2010-03-11,2020-12-31,2022-01-01,2022-12-31
2,release64_current_target,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2010-03-11,2022-12-31,2024-01-01,2024-12-31
3,bull55_hold3,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2218,0,252,True,,2010-03-11,2018-12-31,2020-01-01,2020-12-31
4,bull55_hold3,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,253,0,252,True,,2010-03-11,2020-12-31,2022-01-01,2022-12-31
5,bull55_hold3,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2010-03-11,2022-12-31,2024-01-01,2024-12-31
6,bull65_hold3,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2218,0,252,True,,2010-03-11,2018-12-31,2020-01-01,2020-12-31
7,bull65_hold3,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,253,0,252,True,,2010-03-11,2020-12-31,2022-01-01,2022-12-31
8,bull65_hold3,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2010-03-11,2022-12-31,2024-01-01,2024-12-31
9,bull70_hold5,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2218,0,252,True,,2010-03-11,2018-12-31,2020-01-01,2020-12-31


,candidate,fold,window,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,release64_current_target,wf_2020_train,wf_2020_train,2470,42,873,30,9,9,0.085957,0.085957,0.568200,0.568200,True,True,risk,1.000000e-09
1,release64_current_target,wf_2022_train,wf_2022_train,505,13,234,9,1,2,0.015671,0.031056,0.203723,0.203723,True,True,risk,1.000000e-09
2,release64_current_target,wf_2023_2024_train,wf_2023_2024_train,756,165,478,10,152,152,0.181708,0.181708,0.503723,0.503723,True,True,risk,1.000000e-09
3,bull55_hold3,wf_2020_train,wf_2020_train,2470,62,873,30,29,29,0.167535,0.167535,0.568200,0.568200,True,True,risk,1.000000e-09
4,bull55_hold3,wf_2022_train,wf_2022_train,505,13,234,9,1,2,0.031056,0.046440,0.403723,0.403723,True,True,risk,1.000000e-09
5,bull55_hold3,wf_2023_2024_train,wf_2023_2024_train,756,165,478,10,152,152,0.181708,0.181708,0.503723,0.503723,True,True,risk,1.000000e-09
6,bull65_hold3,wf_2020_train,wf_2020_train,2470,62,873,30,29,29,0.199794,0.199794,0.568200,0.568200,True,True,risk,1.000000e-09
7,bull65_hold3,wf_2022_train,wf_2022_train,505,13,234,9,1,2,0.038748,0.054133,0.503723,0.503723,True,True,risk,1.000000e-09
8,bull65_hold3,wf_2023_2024_train,wf_2023_2024_train,756,165,478,10,152,152,0.181708,0.181708,0.503723,0.503723,True,True,risk,1.000000e-09
9,bull70_hold5,wf_2020_train,wf_2020_train,2470,96,873,30,66,66,0.247858,0.247858,0.618200,0.618200,True,True,risk,1.000000e-09


In [4]:

init_qlib(QLIB_DATA_PATH)
replay_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    print("fold", fold)
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, "train")
    test_df = segment_frame(dataset, "test")
    price_frame = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    test_price_series = price_frame.loc[spec["test"][0] : spec["test"][1]]

    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    for candidate, variant_targets in target_variants.items():
        print("  candidate", candidate)
        fold_targets = variant_targets[variant_targets["fold"].astype(str).eq(spec["target_fold"])].copy()
        fold_targets = fold_targets[fold_targets["datetime"].between(train_start, train_end)].copy()
        fold_targets = fold_targets[fold_targets["training_label_allowed"].fillna(False).astype(bool)].copy()
        bull_state = fold_targets.get("bull_reentry_target_state", pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool)

        set_seed(SEED)
        fit = fit_state_objective_bc_actor_from_frame(
            train_df,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=SELECTED_WEIGHT_CONFIG,
            sample_size=0,
            rng_seed=SEED,
        )
        metrics = evaluate_actor_path(
            fit.actor,
            test_df,
            test_price_series,
            profile,
            spec["test"],
            fold=fold,
            candidate=candidate,
        )
        row = {
            "candidate": candidate,
            "fold": fold,
            "target_rows": int(len(fold_targets)),
            "bull_reentry_target_rows": int(bull_state.sum()),
            "bc_samples": int(fit.metrics["samples"]),
            "bc_after_accuracy": float(fit.metrics["after_accuracy"]),
            "bc_after_loss": float(fit.metrics["after_loss"]),
            "return_pct": float(metrics.get("return", float("nan"))),
            "max_dd_pct": float(metrics.get("max_dd", float("nan"))),
            "avg_pos": float(metrics.get("avg_pos", float("nan"))),
            "buy_and_hold_return_pct": float(metrics.get("buy_and_hold_return", float("nan"))),
            "relative_return_pp": float(metrics.get("relative_return", float("nan"))),
            "daily_path": metrics.get("daily_path"),
            "daily_path_rows": metrics.get("daily_path_rows"),
            "metric_basis": metrics.get("metric_basis"),
            "raw_backtest_return": metrics.get("raw_backtest_return"),
            "raw_backtest_max_dd": metrics.get("raw_backtest_max_dd"),
            "metric_return_abs_diff": metrics.get("metric_return_abs_diff"),
            "metric_max_dd_abs_diff": metrics.get("metric_max_dd_abs_diff"),
        }
        row["wf_gate_20dd"] = bool(row["relative_return_pp"] >= TARGET_RELATIVE_RETURN and row["max_dd_pct"] <= MAX_DD_PCT)
        row["strict_gate_15dd"] = bool(row["relative_return_pp"] >= TARGET_RELATIVE_RETURN and row["max_dd_pct"] <= STRICT_MAX_DD_PCT)
        row["relative_gap_pp"] = max(0.0, TARGET_RELATIVE_RETURN - row["relative_return_pp"])
        row["maxdd_gap_20pp"] = max(0.0, row["max_dd_pct"] - MAX_DD_PCT)
        row["maxdd_gap_15pp"] = max(0.0, row["max_dd_pct"] - STRICT_MAX_DD_PCT)
        replay_rows.append(row)

replay_metrics = pd.DataFrame(replay_rows)
replay_metrics.to_csv(CARD / "bc_official_replay_metrics.csv", index=False)
display(replay_metrics)


[86522:MainThread](2026-06-15 04:36:12,838) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86522:MainThread](2026-06-15 04:36:13,306) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86522:MainThread](2026-06-15 04:36:13,307) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


fold wf_2020


[86522:MainThread](2026-06-15 04:36:14,141) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[86522:MainThread](2026-06-15 04:37:00,291) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[86522:MainThread](2026-06-15 04:37:00,291) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,candidate,fold,target_rows,bull_reentry_target_rows,bc_samples,bc_after_accuracy,bc_after_loss,return_pct,max_dd_pct,avg_pos,...,metric_basis,raw_backtest_return,raw_backtest_max_dd,metric_return_abs_diff,metric_max_dd_abs_diff,wf_gate_20dd,strict_gate_15dd,relative_gap_pp,maxdd_gap_20pp,maxdd_gap_15pp
0,release64_current_target,wf_2020,2218,0,1904,0.653887,1.246048,80.375210,21.106156,0.425202,...,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,False,False,7.718361,1.106156,6.106156
1,bull55_hold3,wf_2020,2218,14,1904,0.657038,1.243742,80.375210,21.106156,0.425202,...,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,False,False,7.718361,1.106156,6.106156
2,bull65_hold3,wf_2020,2218,14,1904,0.657038,1.243742,80.375210,21.106156,0.425202,...,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,False,False,7.718361,1.106156,6.106156
3,bull70_hold5,wf_2020,2218,42,1904,0.659664,1.243154,70.042632,20.939130,0.345946,...,official_daily_path_rebased_eval_window,70.042632,20.939130,0.0,3.552714e-15,False,False,18.050938,0.939130,5.939130
4,release64_current_target,wf_2022,253,0,252,0.726190,1.174679,-23.837432,26.860565,0.144199,...,official_daily_path_rebased_eval_window,-23.837432,26.860565,0.0,0.000000e+00,False,False,0.000000,6.860565,11.860565
5,bull55_hold3,wf_2022,253,1,252,0.698413,1.208709,-32.528031,34.111895,0.147164,...,official_daily_path_rebased_eval_window,-32.528031,34.111895,0.0,0.000000e+00,False,False,0.000000,14.111895,19.111895
6,bull65_hold3,wf_2022,253,1,252,0.789683,1.115628,-14.137212,15.072639,0.087837,...,official_daily_path_rebased_eval_window,-14.137212,15.072639,0.0,5.329071e-15,True,False,0.000000,0.000000,0.072639
7,bull70_hold5,wf_2022,253,1,252,0.789683,1.115628,-14.137212,15.072639,0.087837,...,official_daily_path_rebased_eval_window,-14.137212,15.072639,0.0,5.329071e-15,True,False,0.000000,0.000000,0.072639
8,release64_current_target,wf_2023_2024,756,0,755,0.715232,1.187326,13.945428,13.959434,0.197766,...,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,False,False,15.697938,0.000000,0.000000
9,bull55_hold3,wf_2023_2024,756,0,755,0.715232,1.187326,13.945428,13.959434,0.197766,...,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,False,False,15.697938,0.000000,0.000000


In [5]:

def summarize_candidate(group: pd.DataFrame) -> dict[str, Any]:
    gate_count_20 = int(group["wf_gate_20dd"].fillna(False).astype(bool).sum())
    gate_count_15 = int(group["strict_gate_15dd"].fillna(False).astype(bool).sum())
    relative_gap = pd.to_numeric(group["relative_gap_pp"], errors="coerce").fillna(0.0)
    maxdd_gap20 = pd.to_numeric(group["maxdd_gap_20pp"], errors="coerce").fillna(0.0)
    maxdd_gap15 = pd.to_numeric(group["maxdd_gap_15pp"], errors="coerce").fillna(0.0)
    total_gate_loss_20 = float(relative_gap.sum() + 5.0 * maxdd_gap20.sum())
    total_gate_loss_15 = float(relative_gap.sum() + 5.0 * maxdd_gap15.sum())
    worst_idx = (relative_gap + 5.0 * maxdd_gap20).idxmax()
    worst = group.loc[worst_idx]
    return {
        "candidate": str(group["candidate"].iloc[0]),
        "fold_count": int(group["fold"].nunique()),
        "gate_count_20dd": gate_count_20,
        "gate_count_15dd": gate_count_15,
        "full_gate_20dd": bool(gate_count_20 == len(WALK_FORWARD_WINDOWS)),
        "full_gate_15dd": bool(gate_count_15 == len(WALK_FORWARD_WINDOWS)),
        "min_relative_return_pp": float(pd.to_numeric(group["relative_return_pp"], errors="coerce").min()),
        "max_max_dd_pct": float(pd.to_numeric(group["max_dd_pct"], errors="coerce").max()),
        "mean_avg_pos": float(pd.to_numeric(group["avg_pos"], errors="coerce").mean()),
        "total_gate_loss_20dd": total_gate_loss_20,
        "total_gate_loss_15dd": total_gate_loss_15,
        "worst_fold": str(worst["fold"]),
        "worst_fold_relative_return_pp": float(worst["relative_return_pp"]),
        "worst_fold_max_dd_pct": float(worst["max_dd_pct"]),
    }

candidate_summary = pd.DataFrame([summarize_candidate(g) for _, g in replay_metrics.groupby("candidate", sort=False)])
candidate_summary = candidate_summary.sort_values(
    ["full_gate_20dd", "gate_count_20dd", "total_gate_loss_20dd", "total_gate_loss_15dd", "candidate"],
    ascending=[False, False, True, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)

baseline_id = "release64_current_target"
baseline_rows = replay_metrics[replay_metrics["candidate"].eq(baseline_id)].set_index("fold")
delta_rows = []
for _, row in replay_metrics.iterrows():
    if row["candidate"] == baseline_id:
        continue
    base = baseline_rows.loc[row["fold"]]
    delta_rows.append({
        "candidate": row["candidate"],
        "fold": row["fold"],
        "relative_delta_pp": float(row["relative_return_pp"] - base["relative_return_pp"]),
        "return_delta_pct": float(row["return_pct"] - base["return_pct"]),
        "maxdd_delta_pct": float(row["max_dd_pct"] - base["max_dd_pct"]),
        "avg_pos_delta": float(row["avg_pos"] - base["avg_pos"]),
        "candidate_wf_gate_20dd": bool(row["wf_gate_20dd"]),
        "baseline_wf_gate_20dd": bool(base["wf_gate_20dd"]),
    })
delta_vs_baseline = pd.DataFrame(delta_rows)
delta_vs_baseline.to_csv(CARD / "candidate_vs_release64_current_delta.csv", index=False)

selected = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
baseline_summary = candidate_summary[candidate_summary["candidate"].eq(baseline_id)].iloc[0].to_dict()
selected_improves = bool(selected and float(selected["total_gate_loss_20dd"]) < float(baseline_summary["total_gate_loss_20dd"]))
if selected and bool(selected["full_gate_20dd"]):
    decision = "bull_reentry_participation_objective_official_replay_passed_no_promotion"
    next_mechanism = "run_fixed_split_ytd_dashboard_retention_gate_before_actor_promotion"
elif selected_improves:
    decision = "bull_reentry_participation_objective_partial_improvement_no_promotion"
    next_mechanism = "convert_best_participation_pattern_to_checkpoint_or_reward_objective_with_drawdown_guard"
else:
    decision = "bull_reentry_participation_objective_failed_no_promotion"
    next_mechanism = "stop_bull_reentry_target_lift_and_revise_state_representation_or_checkpoint_objective"

selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "selected_candidate": selected.get("candidate", ""),
    "baseline_candidate": baseline_id,
    "selected_improves_gate_loss": selected_improves,
    "all_selected_gate_20dd": bool(selected.get("full_gate_20dd", False)),
    "all_selected_gate_15dd": bool(selected.get("full_gate_15dd", False)),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_training_objective_repair",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_release_gate_bc_official_replay_v1",
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["target", "model_selection", "official_replay"],
    "external_reference": "converted SOXLBacktest qlib buy/sell baseline plus release16 official replay handoff",
    "hypothesis": "Fold-local bull-reentry target lifts can repair release16 underparticipation without breaking drawdown folds.",
    "acceptance_criteria": [
        "target scope audit passes for fold-local train labels",
        "candidate target variants are replayed on wf_2020, wf_2022, and wf_2023_2024",
        "full gate uses relative_return >= 30pp and MaxDD <= 20% on every fold",
        "strict <=15% MaxDD context is retained",
        "no actor is promoted or retained from this diagnostic replay",
    ],
    "expected_tradeoff": "More bull/reentry participation may improve return gaps but can raise MaxDD, especially in 2020 or 2022.",
    "failure_mode": "Bull-reentry target lift may be too sparse or may worsen drawdown, implying the next lever is checkpoint/reward objective or state representation rather than another target floor.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": "release16_release_gate_bc_weight_screen_v1 fixed sparse release-gate BC fit; older bull_reentry replay passed wf_2022 and wf_2023_2024 but not wf_2020.",
    "closest_historical_failure_case": "release16_release_gate_bc_official_replay_v1 fixed slice fit but had 0/3 official gate passes, and older bull_reentry_three_fold_checkpoint_metric_replay_v1 failed wf_2020 return.",
    "why_not_duplicate": "This applies bull-reentry target variants to the current release16 distillation contract and scores official warm-start daily paths with the selected release64 BC weighting.",
    "handoff": handoff,
    "source_replay_decision": source_replay_summary.get("decision"),
    "selected_candidate": selected.get("candidate", ""),
    "selected_summary": selected,
    "baseline_summary": baseline_summary,
    "candidate_summary": candidate_summary.to_dict("records"),
    "target_variant_summary": target_summary.to_dict("records"),
    "gate_summary": {
        "candidate_count": int(candidate_summary["candidate"].nunique()) if not candidate_summary.empty else 0,
        "full_gate_count_20dd": int(candidate_summary["full_gate_20dd"].fillna(False).astype(bool).sum()) if not candidate_summary.empty else 0,
        "full_gate_count_15dd": int(candidate_summary["full_gate_15dd"].fillna(False).astype(bool).sum()) if not candidate_summary.empty else 0,
        "best_gate_count_20dd": int(selected.get("gate_count_20dd", 0)) if selected else 0,
        "best_total_gate_loss_20dd": float(selected.get("total_gate_loss_20dd", float("nan"))) if selected else float("nan"),
        "baseline_total_gate_loss_20dd": float(baseline_summary.get("total_gate_loss_20dd", float("nan"))),
    },
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "output_paths": {
        "target_config_grid": str((CARD / "target_config_grid.csv").relative_to(ROOT)),
        "bc_weight_config": str((CARD / "bc_weight_config.csv").relative_to(ROOT)),
        "target_variant_summary": str((CARD / "target_variant_summary.csv").relative_to(ROOT)),
        "target_scope_audit": str((CARD / "target_scope_audit.csv").relative_to(ROOT)),
        "target_lift_audit": str((CARD / "target_lift_audit.csv").relative_to(ROOT)),
        "bc_official_replay_metrics": str((CARD / "bc_official_replay_metrics.csv").relative_to(ROOT)),
        "candidate_summary": str((CARD / "candidate_summary.csv").relative_to(ROOT)),
        "candidate_vs_release64_current_delta": str((CARD / "candidate_vs_release64_current_delta.csv").relative_to(ROOT)),
        "soxlbacktest_reference_metrics": str((CARD / "soxlbacktest_reference_metrics.csv").relative_to(ROOT)),
        "official_daily_paths": str(DAILY_DIR.relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
    },
    "next_mechanism": next_mechanism,
    "conclusion": (
        "A bull-reentry participation target variant passed all official walk-forward gates, but promotion remains blocked until fixed/YTD/dashboard/retention checks run."
        if decision.endswith("passed_no_promotion")
        else (
            "Bull-reentry participation target variants improved the official gate loss but still did not produce a retained model; convert the useful pattern into checkpoint/reward objective with explicit drawdown guard."
            if selected_improves
            else "Bull-reentry participation target variants did not improve the release64 official replay baseline; stop this target-lift path and revise representation/checkpoint objective."
        )
    ),
}
write_json(CARD / "summary.json", summary)
print("decision", decision)
display(candidate_summary)
display(delta_vs_baseline)


decision bull_reentry_participation_objective_partial_improvement_no_promotion


,candidate,fold_count,gate_count_20dd,gate_count_15dd,full_gate_20dd,full_gate_15dd,min_relative_return_pp,max_max_dd_pct,mean_avg_pos,total_gate_loss_20dd,total_gate_loss_15dd,worst_fold,worst_fold_relative_return_pp,worst_fold_max_dd_pct
0,bull65_hold3,3,1,0,False,False,14.302062,21.106156,0.236935,28.947079,54.310272,wf_2023_2024,14.302062,13.959434
1,bull70_hold5,3,1,0,False,False,11.949062,20.939130,0.210516,38.444524,63.807718,wf_2020,11.949062,20.939130
2,release64_current_target,3,0,0,False,False,14.302062,26.860565,0.255722,63.249902,113.249902,wf_2022,62.750639,26.860565
3,bull55_hold3,3,0,0,False,False,14.302062,34.111895,0.256711,99.506552,149.506552,wf_2022,54.060041,34.111895


,candidate,fold,relative_delta_pp,return_delta_pct,maxdd_delta_pct,avg_pos_delta,candidate_wf_gate_20dd,baseline_wf_gate_20dd
0,bull55_hold3,wf_2020,0.000000,0.000000,0.000000,0.000000,False,False
1,bull65_hold3,wf_2020,0.000000,0.000000,0.000000,0.000000,False,False
2,bull70_hold5,wf_2020,-10.332577,-10.332577,-0.167026,-0.079256,False,False
3,bull55_hold3,wf_2022,-8.690598,-8.690598,7.251330,0.002966,False,False
4,bull65_hold3,wf_2022,9.700221,9.700221,-11.787926,-0.056362,True,False
5,bull70_hold5,wf_2022,9.700221,9.700221,-11.787926,-0.056362,True,False
6,bull55_hold3,wf_2023_2024,0.000000,0.000000,0.000000,0.000000,False,False
7,bull65_hold3,wf_2023_2024,0.000000,0.000000,0.000000,0.000000,False,False
8,bull70_hold5,wf_2023_2024,0.000000,0.000000,0.000000,0.000000,False,False


In [6]:

summary = load_json(CARD / "summary.json")
lines = [
    f"Decision: `{summary['decision']}`.",
    f"Selected candidate: `{summary['selected_candidate']}`.",
    f"Best gate count <=20% MaxDD: {summary['gate_summary']['best_gate_count_20dd']} / {len(WALK_FORWARD_WINDOWS)}.",
    f"Best total gate loss <=20% MaxDD: {summary['gate_summary']['best_total_gate_loss_20dd']:.2f}; baseline: {summary['gate_summary']['baseline_total_gate_loss_20dd']:.2f}.",
    f"Next mechanism: `{summary['next_mechanism']}`.",
    summary["conclusion"],
]
display(Markdown("\n".join(f"- {line}" for line in lines)))


- Decision: `bull_reentry_participation_objective_partial_improvement_no_promotion`.
- Selected candidate: `bull65_hold3`.
- Best gate count <=20% MaxDD: 1 / 3.
- Best total gate loss <=20% MaxDD: 28.95; baseline: 63.25.
- Next mechanism: `convert_best_participation_pattern_to_checkpoint_or_reward_objective_with_drawdown_guard`.
- Bull-reentry participation target variants improved the official gate loss but still did not produce a retained model; convert the useful pattern into checkpoint/reward objective with explicit drawdown guard.